In [2]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.


In [17]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Fake data: 1000 samples, 10 features each
X = torch.randn(1000, 10)              # features
y = torch.randint(0, 2, (1000, 1)).float()   # binary labels (float for BCE)

dataset = TensorDataset(X, y)

train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,       # shuffle every epoch (good for training)
    num_workers=0,      # 0 = load in main process (safe on CPU / Colab)
    drop_last=False,
)

print(len(train_loader))
train_loader   # 1000 / 32 ≈ 32 batches

32


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split

device = torch.device("cpu")

# ---------------------------------------------------------------
# 1. DATA — replace this with your real data
# ---------------------------------------------------------------
X = torch.randn(1000, 10)                       # 1000 samples, 10 features
y = torch.randint(0, 2, (1000, 1)).float()      # binary labels as floats

full_ds = TensorDataset(X, y)

n_train = int(0.8 * len(full_ds))
train_ds, val_ds = random_split(full_ds, [n_train, len(full_ds) - n_train])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

# ---------------------------------------------------------------
# 2. MODEL
# ---------------------------------------------------------------
model = nn.Sequential(
    nn.Linear(10, 5), nn.ReLU(),
    nn.Linear(5, 5),  nn.ReLU(),
    nn.Linear(5, 1),                 # 1 logit → binary classification
).to(device)

losscalc  = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

# ---------------------------------------------------------------
# 3. TRAIN LOOP
# ---------------------------------------------------------------
for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs  = inputs.to(device)
        targets = targets.to(device).float().view(-1, 1)

        optimizer.zero_grad(set_to_none=True)

        output = model(inputs)
        loss   = losscalc(output, targets)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()      # ← you were missing this

    avg_train_loss = running_train_loss / len(train_loader)

    # ---- validation ----
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs  = inputs.to(device)
            targets = targets.to(device).float().view(-1, 1)
            output  = model(inputs)
            loss    = losscalc(output, targets)
            running_val_loss += loss.item()

    avg_val_loss = running_val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}]  "
          f"train_loss={avg_train_loss:.4f}  val_loss={avg_val_loss:.4f}")

Epoch [1/10]  train_loss=0.7045  val_loss=0.6870
Epoch [2/10]  train_loss=0.7008  val_loss=0.6874
Epoch [3/10]  train_loss=0.6982  val_loss=0.6880
Epoch [4/10]  train_loss=0.6965  val_loss=0.6893
Epoch [5/10]  train_loss=0.6952  val_loss=0.6897
Epoch [6/10]  train_loss=0.6942  val_loss=0.6911
Epoch [7/10]  train_loss=0.6935  val_loss=0.6916
Epoch [8/10]  train_loss=0.6928  val_loss=0.6927
Epoch [9/10]  train_loss=0.6921  val_loss=0.6930
Epoch [10/10]  train_loss=0.6916  val_loss=0.6929
